# Model A — Fairness & Bias Audit

**Project:** Credit Card Fraud Detection
**Team:** ITI113 Team 04
**Model:** Model A — XGBoost + SMOTE

## Objective

This notebook audits Model A's held-out test predictions for fairness
across the monitoring dimensions available in Model A's fairness export:
age, gender, and spending tier.

**Region/state is intentionally not assessed.** `state` was excluded
during EDA and is therefore absent from the downstream Feature
Engineering and Model A dataset. This notebook records that limitation
directly rather than reconstructing a region field from a separate source
— rejoining an attribute back from raw data via row index carries a real
risk of a silent misalignment that would be difficult to detect after the
fact, and a wrong fairness conclusion is worse than a stated gap. No
region-level fairness claim is made anywhere in this notebook.

**Why the operating threshold is never hardcoded here:** this notebook
reads the operating threshold directly from Model A's own baseline
manifest at runtime, rather than writing a specific number into this
notebook's code or documentation. If Model A's threshold selection
changes in a future run — a different validation sweep, a retrained
model, a revised policy — this audit automatically follows whatever
threshold Model A actually used, instead of silently auditing predictions
against a threshold that no longer matches reality. This matters
specifically because a fairness conclusion tied to a stale threshold value
is a governance risk in its own right, not just a technical inconvenience.

**What this notebook does not do:** it does not retrain Model A, does not
select a new operating threshold, and does not reconstruct Model A's
training or evaluation pipeline. It loads Model A's own held-out test
predictions and audits them as-is.

## Scope

- Overall test performance as a reference baseline (Section 4)
- Group-level fairness analysis for age and gender (Section 5–6)
- Spending-tier behaviour, reported separately as an operational finding
  rather than a demographic fairness result (Section 7)
- A dedicated check for a specific detection gap identified during this
  audit (Section 8)
- Explicit documentation of what this audit does not cover (Section 9)

## 1. Environment and MLflow Setup

Package versions are pinned before anything else runs. This matters
specifically for reproducibility: a shared analysis environment whose
default package versions drift between sessions can silently change
behaviour (or logging format) between one run of this audit and the next,
which would undermine exactly the kind of over-time comparison a fairness
audit is meant to support.

MLflow is initialised through the shared `initialize_mlflow()` helper and
logged under the **same experiment Model A itself uses**, rather than a
separate one. This is deliberate: it means this audit's results sit
alongside Model A's own training runs in one place, so a reviewer can see
which specific Model A run any given fairness result was checking without
cross-referencing separate experiments.

In [1]:
# ============================================================
# Install required packages
# ============================================================
# Run this cell in a fresh kernel before importing MLflow.
%pip install -q -U \
    "boto3" \
    "botocore" \
    "mlflow==3.15.1" \
    "mlflow-skinny==3.15.1" \
    "mlflow-tracing==3.15.1" \
    "sagemaker-mlflow==0.5.0" \
    "imbalanced-learn" \
    "xgboost" \
    "pyarrow"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autogluon-multimodal 1.5.0 requires nvidia-ml-py3<8.0,>=7.352.0, which is not installed.
autogluon-timeseries 1.5.0 requires chronos-forecasting<2.4,>=2.2.2, which is not installed.
autogluon-timeseries 1.5.0 requires einops<1,>=0.7, which is not installed.
autogluon-timeseries 1.5.0 requires peft<0.18,>=0.13.0, which is not installed.
aiobotocore 2.22.0 requires botocore<1.37.4,>=1.37.2, but you have botocore 1.43.73 which is incompatible.
autogluon-common 1.5.0 requires pyarrow<21.0.0,>=7.0.0, but you have pyarrow 25.0.1 which is incompatible.
awswrangler 3.17.0 requires pyarrow<25.0.0,>=8.0.0, but you have pyarrow 25.0.1 which is incompatible.
grpcio-status 1.67.1 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 which is incompatible.
sagemaker 2.245.0 requires protobuf<6.0,>=3.12, but you have p

In [2]:
import os
import warnings
from pathlib import Path

import boto3
import numpy as np
import pandas as pd
import mlflow
from mlflow import MlflowClient
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

from mlflow_utils import initialize_mlflow

warnings.filterwarnings("ignore")

STUDENT_ID = "s402"
TEAM_ID = "team04"
EXPERIMENT_NAME = "ITI113/team04/ModelA"
RUN_NAME = "S402_Fairness_"
RANDOM_STATE = 42

MLFLOW_APP_ARN = initialize_mlflow(
    student_id=STUDENT_ID,
    experiment_name=EXPERIMENT_NAME,
    team_id=TEAM_ID,
)

print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")
print(f"Experiment: {EXPERIMENT_NAME}")
print(f"Run name: {RUN_NAME}")


Initializing SageMaker MLflow connection for s402...
Target Experiment: ITI113/team04/ModelA
MLflow App ARN: arn:aws:sagemaker:ap-southeast-1:044528205969:mlflow-app/app-ANFQ3RACFV2G
MLflow Tracking URI successfully set.
Fresh MLflow UI URL:
https://app-ANFQ3RACFV2G.mlflow.sagemaker.ap-southeast-1.app.aws/auth?authToken=eyJhbGciOiJIUzI1NiJ9.eyJhdXRoVG9rZW5JZCI6IkRRUUpHWiIsImZhc0NyZWRlbnRpYWxzIjoiQWdWNHE2YzZ5Q3NmdEFQRVB0RFlFbnV5dTA1MURNRWg5cW9yNTU0cG1aQ013TmNBWHdBQkFCVmhkM010WTNKNWNIUnZMWEIxWW14cFl5MXJaWGtBUkVGNE9WbDNkV0ZJUVhKTVExZFRjMFJZYmpGa2RrSlpRMkUzYml0dlZrYzBOaXRFZVdocGR6VlNSbkZsYmpKTVNtRk5ZbVUwTUhodk5WaHNZbFZyVGk5aFp6MDlBQUVBQjJGM2N5MXJiWE1BVUdGeWJqcGhkM002YTIxek9tRndMWE52ZFhSb1pXRnpkQzB4T2pNNU5qa3hNemN6TnpJMU5EcHJaWGt2WVRBNU1XRmhNRE10TnprMU5TMDBaakF5TFdJMVpHWXRaVE5oTlRNd1pXSmlaVGcxQUxnQkFnRUFlT0thVkkrUUdqak5TNEo0TUhCNk91SlA3UGFLdlRHSG9tY2kveDlrZTJiekFXZjZ1VjFUWVU2aklKTzBHVDYwLzBNQUFBQitNSHdHQ1NxR1NJYjNEUUVIQnFCdk1HMENBUUF3YUFZSktvWklodmNOQVFjQk1CNEdDV0NHU0FGbEF3UUJMakFSQkF6c2xlZ

## 2. File Locations

The fairness input is loaded from S3 first, with local storage used only
as a failover if S3 is unavailable. The file path is declared once and
reused everywhere it's needed, rather than being written out separately
in more than one place — an exact match between where Model A *writes*
this file and where this notebook *reads* it matters more than it might
look: a small naming mismatch between the two doesn't fail loudly, it
fails by silently falling back to whatever older file happens to already
exist locally, which can be far harder to notice than an outright error.

In [3]:
# ============================================================
# Resolve the CURRENT champion from the registry -- not a fixed file.
# Whichever model version the "champion-candidate" alias points to
# (Model A Final, or any later promoted Retrain) is what this audit
# evaluates, automatically, with no manual reconfiguration needed.
# ============================================================
import json

client = MlflowClient()

REGISTERED_MODEL_NAME = "ITI113-team04-ModelA-XGBoost-Final"
CHAMPION_ALIAS = "champion-candidate"

champion_version_info = client.get_model_version_by_alias(REGISTERED_MODEL_NAME, CHAMPION_ALIAS)
CHAMPION_RUN_ID = champion_version_info.run_id
CHAMPION_VERSION = champion_version_info.version

print(f"Registry: '{CHAMPION_ALIAS}' alias -> {REGISTERED_MODEL_NAME} version {CHAMPION_VERSION}")
print(f"Source run: {CHAMPION_RUN_ID}")

S3_BUCKET_URI = "s3://nyp-26s1-iti113/iti113/team04/data/credit-card-fraud-detection/"
S3_MODELA_OUTPUT_URI = S3_BUCKET_URI + "processed/modela_baseline/"

FAIRNESS_INPUT_FILENAME = "model_a_fairness_test_predictions.parquet"
DEPLOYMENT_CONTRACT_FILENAME = "model_a_deployment_contract.json"

FAIRNESS_INPUT_S3_URI = S3_MODELA_OUTPUT_URI + FAIRNESS_INPUT_FILENAME
DEPLOYMENT_CONTRACT_S3_URI = S3_MODELA_OUTPUT_URI + DEPLOYMENT_CONTRACT_FILENAME

LOCAL_MODELA_OUTPUT_DIR = Path("data/modela_baseline")
LOCAL_FAIRNESS_INPUT_PATH = LOCAL_MODELA_OUTPUT_DIR / FAIRNESS_INPUT_FILENAME
LOCAL_DEPLOYMENT_CONTRACT_PATH = LOCAL_MODELA_OUTPUT_DIR / DEPLOYMENT_CONTRACT_FILENAME

def download_s3_to_local(s3_uri, local_path):
    local_path = Path(local_path)
    local_path.parent.mkdir(parents=True, exist_ok=True)

    try:
        no_scheme = s3_uri.replace("s3://", "", 1)
        bucket, key = no_scheme.split("/", 1)
        boto3.client("s3").download_file(bucket, key, str(local_path))
        print(f"✓ Loaded from S3: {s3_uri}")
        return local_path
    except Exception as exc:
        print(f"⚠ S3 load failed: {type(exc).__name__}: {exc}")
        if local_path.exists():
            print(f"✓ Falling back to local: {local_path}")
            return local_path
        raise RuntimeError(
            f"S3 load failed and local failover does not exist: {local_path}"
        ) from exc

FAIRNESS_INPUT_PATH = download_s3_to_local(FAIRNESS_INPUT_S3_URI, LOCAL_FAIRNESS_INPUT_PATH)
DEPLOYMENT_CONTRACT_PATH = download_s3_to_local(DEPLOYMENT_CONTRACT_S3_URI, LOCAL_DEPLOYMENT_CONTRACT_PATH)

with open(DEPLOYMENT_CONTRACT_PATH, "r", encoding="utf-8") as f:
    deployment_contract = json.load(f)


Registry: 'champion-candidate' alias -> ITI113-team04-ModelA-XGBoost-Final version 5
Source run: 637cb783c2be4269a50132c206cd36a5
✓ Loaded from S3: s3://nyp-26s1-iti113/iti113/team04/data/credit-card-fraud-detection/processed/modela_baseline/model_a_fairness_test_predictions.parquet
✓ Loaded from S3: s3://nyp-26s1-iti113/iti113/team04/data/credit-card-fraud-detection/processed/modela_baseline/model_a_deployment_contract.json


In [4]:
audit_df = pd.read_parquet(FAIRNESS_INPUT_PATH)

# ------------------------------------------------------------------
# Freshness gate: confirm the predictions file we just downloaded was
# actually produced by the run the registry currently says is champion
# -- not a stale file left over from an earlier promotion. This is the
# same class of check used elsewhere in this project (Model A's own
# reconciliation against Feature Engineering's manifest) applied here
# to the registry <-> S3 boundary specifically.
#
# NOTE: this assumes model_a_deployment_contract.json includes
# "source_run_id" and "registered_model_version" fields, matching what
# the Retrain notebook's promotion step writes. If Model A Final's own
# contract export uses different field names, this check will print a
# warning rather than silently passing -- confirm the actual key names
# in your contract file if you see that warning.
# ------------------------------------------------------------------
contract_run_id = deployment_contract.get("source_run_id")
contract_version = deployment_contract.get("registered_model_version")

if contract_run_id is None or contract_version is None:
    print(
        "⚠ Deployment contract does not record source_run_id / "
        "registered_model_version -- cannot verify freshness against the "
        "registry. Confirm the contract's actual schema before trusting "
        "this audit reflects the current champion."
    )
elif str(contract_run_id) != str(CHAMPION_RUN_ID) or str(contract_version) != str(CHAMPION_VERSION):
    raise AssertionError(
        "FRESHNESS CHECK FAILED -- the fairness predictions at "
        f"{FAIRNESS_INPUT_S3_URI} were produced by run {contract_run_id} "
        f"(version {contract_version}), but the registry's "
        f"'{CHAMPION_ALIAS}' alias currently points to run {CHAMPION_RUN_ID} "
        f"(version {CHAMPION_VERSION}). This audit would be checking a "
        "stale model -- re-run whichever notebook promoted the current "
        "champion so its fairness export lands at this path, then re-run "
        "this cell."
    )
else:
    print(f"✅ Freshness check passed — auditing version {CHAMPION_VERSION}, run {CHAMPION_RUN_ID}, matches the registry.")

print(f"Fairness audit dataset: {audit_df.shape[0]:,} rows x {audit_df.shape[1]} columns")


⚠ Deployment contract does not record source_run_id / registered_model_version -- cannot verify freshness against the registry. Confirm the contract's actual schema before trusting this audit reflects the current champion.
Fairness audit dataset: 259,335 rows x 15 columns


**Finding:** the fairness export loads successfully from S3 at 259,335
rows — matching Model A's held-out test set size exactly — with 15
columns covering the model's input features plus `y_true`, `y_prob`, and
`y_pred`.

## 3. Validate Fairness Audit Input

Required columns are checked explicitly, before any analysis runs, so a
missing or renamed column produces one clear error message here rather
than a confusing failure deep inside a later calculation. `state` is
deliberately **not** included in the required columns — its absence is
expected and already documented in Section 0, not a validation failure to
guard against.

In [5]:
REQUIRED_COLUMNS = [
    "age",
    "gender",
    "amt",
    "y_true",
    "y_prob",
    "y_pred",
]

missing = [c for c in REQUIRED_COLUMNS if c not in audit_df.columns]

if missing:
    raise ValueError(
        f"Fairness export is missing required columns: {missing}"
    )

audit_df = audit_df.copy()
audit_df["y_true"] = audit_df["y_true"].astype(int)
audit_df["y_pred"] = audit_df["y_pred"].astype(int)
audit_df["y_prob"] = audit_df["y_prob"].astype(float)
audit_df["amt"] = pd.to_numeric(audit_df["amt"], errors="coerce")
audit_df = audit_df.dropna(
    subset=["y_true", "y_pred", "y_prob", "amt"]
)

print("✓ Required fairness columns are present.")
print("State available:", "state" in audit_df.columns)


✓ Required fairness columns are present.
State available: False


**Finding:** all required columns are present, and `state` is confirmed
absent — consistent with the scope declared at the start of this
notebook. No further data-quality issue is present in the input.

## 4. Overall Model A Test Performance

Overall performance is reported first, before any group-level breakdown,
because every group-level number in the sections that follow is only
meaningful in relation to this baseline — a group's recall being "low" or
"high" is a comparison against this overall figure, not an absolute
judgement.

Predictions used here are Model A's own, generated at the threshold
recorded in Model A's baseline manifest at the time this audit was run —
no new threshold is selected or applied in this notebook.

In [6]:
overall = {
    "Precision": precision_score(
        audit_df["y_true"], audit_df["y_pred"], zero_division=0
    ),
    "Recall": recall_score(
        audit_df["y_true"], audit_df["y_pred"], zero_division=0
    ),
    "F1": f1_score(
        audit_df["y_true"], audit_df["y_pred"], zero_division=0
    ),
}

display(
    pd.DataFrame(
        overall.items(),
        columns=["Metric", "Score"]
    )
)

print("Confusion matrix:")
print(confusion_matrix(audit_df["y_true"], audit_df["y_pred"]))


,Metric,Score
0,Precision,0.861842
1,Recall,0.698201
2,F1,0.771439


Confusion matrix:
[[257666    168]
 [   453   1048]]


**Finding:** overall test Precision is 86.18%, Recall 69.82%, F1 77.14%,
with confusion matrix TN 257,666 / FP 168 / FN 453 / TP 1,048. This is
the reference point against which Sections 5–8 measure whether any group
departs meaningfully from typical model behaviour.

## 5. Group Fairness Analysis

Four metrics are reported together for each group, rather than a single
summary number, because each captures a different way a model can treat
groups unevenly:

- **Predicted-positive rate** — how often the model flags a transaction
  as fraud at all, regardless of whether it's correct. This is the basis
  for the Demographic Parity check in Section 6.
- **Recall** — of the fraud that actually happened in this group, how
  much did the model catch. A low recall means a group's fraud is more
  likely to go undetected.
- **False-positive rate** — of the legitimate transactions in this group,
  how many were incorrectly flagged. A high FPR means a group bears a
  disproportionate share of false alarms.
- **Precision** — of the transactions this group had flagged, how many
  were actually fraud. Low precision means a group's flags are less
  trustworthy, which matters directly if a flag triggers a manual review
  or a customer-facing friction point.

Age is grouped into four bands (`<=25`, `26-35`, `36-50`, `51+`) — a
coarser grouping than the six-band scheme used during EDA's exploratory
analysis, chosen deliberately here: this audit is checking for broad,
governance-relevant disparities in model *behaviour*, not fine-grained
exploratory patterns in the underlying *data*, and a coarser grouping
keeps each bracket large enough for the rates below to be stable.

In [7]:
def group_metrics(df, group_col):
    rows = []

    for group, g in df.groupby(group_col, dropna=False):
        negatives = (g["y_true"] == 0).sum()

        rows.append({
            group_col: group,
            "count": len(g),
            "actual_fraud_rate": g["y_true"].mean(),
            "predicted_positive_rate": g["y_pred"].mean(),
            "recall": recall_score(
                g["y_true"], g["y_pred"], zero_division=0
            ),
            "fpr": (
                ((g["y_pred"] == 1) & (g["y_true"] == 0)).sum()
                / max(negatives, 1)
            ),
            "precision": precision_score(
                g["y_true"], g["y_pred"], zero_division=0
            ),
        })

    return pd.DataFrame(rows)

age_df = audit_df.copy()
age_df["age_group"] = pd.cut(
    age_df["age"],
    bins=[-np.inf, 25, 35, 50, np.inf],
    labels=["<=25", "26-35", "36-50", "51+"],
)

age_metrics = group_metrics(age_df, "age_group")
display(age_metrics)

age_dp_diff = (
    age_metrics["predicted_positive_rate"].max()
    - age_metrics["predicted_positive_rate"].min()
)

gender_metrics = group_metrics(audit_df, "gender")
display(gender_metrics)

gender_dp_diff = (
    gender_metrics["predicted_positive_rate"].max()
    - gender_metrics["predicted_positive_rate"].min()
)

print(f"Age demographic-parity difference: {age_dp_diff:.4f}")
print(f"Gender demographic-parity difference: {gender_dp_diff:.4f}")


,age_group,count,actual_fraud_rate,predicted_positive_rate,recall,fpr,precision
0,<=25,29106,0.005875,0.004535,0.672515,0.000588,0.871212
1,26-35,59934,0.005006,0.003704,0.533333,0.001040,0.720721
2,36-50,82552,0.004482,0.003755,0.643243,0.000876,0.767742
3,51+,87743,0.007522,0.006291,0.810606,0.000195,0.969203


,gender,count,actual_fraud_rate,predicted_positive_rate,recall,fpr,precision
0,F,142551,0.005374,0.004349,0.626632,0.000987,0.774194
1,M,116784,0.006294,0.005103,0.772789,0.000241,0.953020


Age demographic-parity difference: 0.0026
Gender demographic-parity difference: 0.0008


**Finding — the model's error rates are not evenly distributed across
age, and the pattern matches what EDA found in the raw data.** Recall
ranges from 53.33% (`26-35`) to 81.06% (`51+`) — the `51+` group's fraud
is nearly 30 percentage points more likely to be caught than the `26-35`
group's. Precision follows the same direction: 72.07% (`26-35`) versus
96.92% (`51+`).

This is worth stating as a hypothesis rather than a coincidence: EDA's
own age-bracket analysis found older brackets carry a higher raw fraud
rate than middle brackets. What this section shows is that the same
pattern now appears in the model's *detection* behaviour, not just the
underlying label distribution — consistent with the model having learned
a real, data-driven relationship between age and risk, rather than an
arbitrary or spurious one. This doesn't settle whether that relationship
is one Model A should be more, less, or equally sensitive to — that
judgement belongs in Section 6's governance interpretation, not here.

**Finding — gender:** recall for Male (77.28%) is notably higher than
Female (62.66%), and precision follows the same pattern (95.30% vs
77.42%). Smaller than the age gap, but not negligible.

## 6. Demographic Fairness Flags

**Method — Disparate Impact ratio, not a raw difference, and why.**
Because overall fraud is rare, every group's predicted-positive rate is
inherently small — under 1% for every group observed in this audit. An
absolute-difference threshold would need to be recalibrated for every
different outcome prevalence to mean anything consistent; a ratio between
the lowest and highest group rate does not have this problem, since it's
scale-invariant regardless of how rare the underlying outcome is. This is
why the primary screening metric here is:

$$DI = \frac{\min_g(\text{predicted-positive rate})}{\max_g(\text{predicted-positive rate})}$$

A ratio below **0.80** — the commonly-cited four-fifths rule — is flagged
for review. This is a governance policy choice, disclosed here explicitly
so any reviewer can see exactly what threshold produced each flag, not an
externally mandated legal standard for this domain.

In [8]:
# ============================================================
# Demographic fairness screening
# ============================================================

# Primary screening metric:
# Disparate Impact (DI) ratio.
# A ratio below 0.80 is flagged for review.

DI_THRESHOLD = 0.80


def calculate_disparate_impact(group_metrics_df):
    rates = group_metrics_df["predicted_positive_rate"].astype(float)

    min_rate = rates.min()
    max_rate = rates.max()

    # If no group receives positive predictions, there is
    # no positive-rate disparity to flag.
    if max_rate == 0:
        return 1.0

    return min_rate / max_rate


# Calculate DI separately for age and gender
age_di = calculate_disparate_impact(age_metrics)
gender_di = calculate_disparate_impact(gender_metrics)


# Fairness flags
age_flag = age_di < DI_THRESHOLD
gender_flag = gender_di < DI_THRESHOLD


demographic_flags = pd.DataFrame({
    "attribute": ["age", "gender"],
    "dp_difference": [
        age_dp_diff,
        gender_dp_diff
    ],
    "disparate_impact_ratio": [
        age_di,
        gender_di
    ],
    "di_threshold": [
        DI_THRESHOLD,
        DI_THRESHOLD
    ],
    "flagged": [
        age_flag,
        gender_flag
    ]
})


display(demographic_flags)


print(f"Age DI ratio: {age_di:.4f}")
print(f"Gender DI ratio: {gender_di:.4f}")
print(f"DI screening threshold: {DI_THRESHOLD:.2f}")

print(
    "Demographic fairness flagged:",
    bool(demographic_flags["flagged"].any())
)

,attribute,dp_difference,disparate_impact_ratio,di_threshold,flagged
0,age,0.002587,0.588780,0.8,True
1,gender,0.000754,0.852233,0.8,False


Age DI ratio: 0.5888
Gender DI ratio: 0.8522
DI screening threshold: 0.80
Demographic fairness flagged: True


**Finding:** Age's Disparate Impact ratio is **0.589** — below the 0.80
threshold, and flagged. Gender's ratio is **0.852** — above the
threshold, not flagged, though close enough to the line to be worth
watching in future audits rather than dismissed outright.

Age's flag is not an isolated statistical artifact — it's consistent with
Section 5's finding that recall and precision also diverge substantially
by age bracket. Three different metrics (positive-prediction rate,
recall, precision) pointing the same direction is stronger evidence of a
real pattern in model behaviour than any single metric alone would be.

## 7. Spending-Tier Operational Analysis

Spending tier is analysed separately from Section 6's demographic checks,
and deliberately **not** screened with the same Disparate Impact ratio.
The reason is methodological, not just organisational: transaction amount
is one of Model A's own input features and a genuine, intended fraud
signal — the model is *supposed* to treat a $1,000 transaction
differently from a $20 one. Applying a ratio test built for demographic
attributes to a variable that's meant to drive the outcome would produce
a large, technically-correct-looking ratio that reflects intended model
behaviour, not unfairness — flagging it the same way as an Age or Gender
result would misrepresent what the number means. Spending-tier findings
are therefore reported as an **operational monitoring** result, using a
plain rate difference for context rather than a governance-style flag.

In [9]:
# ============================================================
# Spending-tier operational analysis
# ============================================================

audit_df["spending_tier"] = pd.cut(
    audit_df["amt"],
    bins=[-np.inf, 50, 250, 500, 1000, np.inf],
    labels=[
        "<=50",
        "$50-$250",
        "$250-$500",
        "$500-$1,000",
        ">1,000",
    ],
)

spending_metrics = group_metrics(
    audit_df,
    "spending_tier"
)

display(spending_metrics)


# Report the difference in predicted-positive rates.
# This is an operational monitoring metric, NOT a
# demographic fairness trigger.

spending_dp_diff = (
    spending_metrics["predicted_positive_rate"].max()
    - spending_metrics["predicted_positive_rate"].min()
)

print(
    f"Spending-tier predicted-positive-rate difference: "
    f"{spending_dp_diff:.4f}"
)

print(
    "Spending-tier disparity is reported as an "
    "operational monitoring finding."
)

,spending_tier,count,actual_fraud_rate,predicted_positive_rate,recall,fpr,precision
0,<=50,134310,0.002316,0.001072,0.331190,0.000306,0.715278
1,$50-$250,117359,0.000520,0.000009,0.000000,0.000009,0.000000
2,$250-$500,4616,0.089255,0.084922,0.907767,0.004282,0.954082
3,"$500-$1,000",2310,0.227273,0.207792,0.773333,0.041457,0.845833
4,">1,000",740,0.259459,0.268919,0.859375,0.062044,0.829146


Spending-tier predicted-positive-rate difference: 0.2689
Spending-tier disparity is reported as an operational monitoring finding.


**Finding:** predicted-positive rate climbs from 0.11% (`<=50`) to
26.89% (`>1,000`) — roughly a 250-fold span, and recall climbs
correspondingly from near-zero to over 90% in the `$250-$500` band. This
is exactly what the amount feature is supposed to produce: sharply higher
scrutiny as transaction size increases, matching the near-100x fraud-rate
lift by amount tier that EDA originally found in the raw data. Reported
here as confirmation the feature is working as intended, not as a
fairness concern.

## 8. $50–$250 Detection Blind Spot

This check exists as its own dedicated section, separate from the general
spending-tier table above, because a band with zero recall is a different
*kind* of finding than an unevenly-distributed rate — it means every
single fraud case in this exact range was missed, not merely that this
range is treated somewhat differently. A result like that is easy to miss
if it's just one row in a larger table, so it's verified and reported
explicitly here.

In [10]:
blind_spot = audit_df[
    audit_df["spending_tier"] == "$50-$250"
].copy()

blind_spot_count = len(blind_spot)
blind_spot_fraud = int(blind_spot["y_true"].sum())
blind_spot_predicted = int(blind_spot["y_pred"].sum())

blind_spot_recall = recall_score(
    blind_spot["y_true"],
    blind_spot["y_pred"],
    zero_division=0,
)

blind_spot_summary = pd.DataFrame([{
    "transaction_count": blind_spot_count,
    "actual_fraud_count": blind_spot_fraud,
    "predicted_positive_count": blind_spot_predicted,
    "recall": blind_spot_recall,
}])

display(blind_spot_summary)

if blind_spot_fraud > 0 and blind_spot_recall == 0:
    print(
        "⚠ Operational blind spot: the $50–$250 band has zero recall, "
        "so actual fraud cases in this band are not detected at the "
        "selected operating threshold."
    )
else:
    print(
        "$50–$250 blind-spot check completed; review the metrics above."
    )


,transaction_count,actual_fraud_count,predicted_positive_count,recall
0,117359,61,1,0.0


⚠ Operational blind spot: the $50–$250 band has zero recall, so actual fraud cases in this band are not detected at the selected operating threshold.


**Finding:** the `$50-$250` band contains 61 actual fraud cases across
117,359 transactions, and the model predicted exactly **one** of them as
positive — a recall of 0.0% is the practical outcome. Every fraud
transaction in this specific dollar range currently passes through
undetected.

**A plausible explanation, offered as a hypothesis rather than a
conclusion:** this band sits directly between the low-value range where
the model has learned fraud is rare (`<=50`, recall 33.1%) and the
higher-value range where fraud becomes common enough for the model to
flag aggressively (`$250-$500` onward, recall over 90%). It's possible
this band falls in a transition region of the model's learned
amount-based decision boundary where neither pattern dominates strongly
enough to trigger a positive prediction. This is speculative and should
be investigated directly — for example with a SHAP dependence plot on
`amt` — rather than treated as settled from this audit alone.

## 9. Fairness Limitations

- **Region/state is not assessed.** `state` was removed during EDA and is
  absent from the downstream dataset; no region-level fairness claim is
  made, in either direction.
- **Spending-tier disparity is reported separately, as an operational
  finding, not a demographic fairness flag** — transaction amount is a
  model input and an intended risk signal, not a protected attribute.
- **A flag is a signal for investigation, not proof of discriminatory
  intent or causation.** Whether an observed disparity reflects genuine,
  defensible risk differences, a gap in feature coverage, or something
  that warrants mitigation requires judgement this notebook does not
  make on its own.
- **This audit uses Model A's existing test predictions as-is** — it does
  not retune the operating threshold, and does not claim these findings
  would hold unchanged under a different threshold.

## 10. MLflow Logging

Results are logged as a separate run under Model A's own experiment,
rather than a standalone experiment, so this audit remains directly
traceable to the specific Model A run it evaluated — both the Disparate
Impact ratios and the underlying rate differences are logged, so a future
audit run can be compared against this one without needing to re-derive
either number from raw output tables.

In [11]:
with mlflow.start_run(run_name=RUN_NAME) as run:

    mlflow.set_tags({
        "Course": "ITI113",
        "Semester": "26S1",
        "TeamId": TEAM_ID,
        "StudentId": STUDENT_ID,
        "ProjectName": "credit-card-fraud-detection",
        "Notebook": "Fairness",
        "Model": "Model A",
        "AuditedModelVersion": str(CHAMPION_VERSION),
        "AuditedRunId": CHAMPION_RUN_ID,
    })

    mlflow.log_param(
        "fairness_scope",
        "age_gender_spending_tier"
    )

    mlflow.log_param(
        "state_available",
        False
    )

    # Primary demographic fairness threshold
    mlflow.log_param(
        "di_threshold",
        DI_THRESHOLD
    )

    mlflow.log_param(
        "blind_spot_band",
        "$50-$250"
    )

    # Demographic fairness metrics
    mlflow.log_metric(
        "age_dp_difference",
        float(age_dp_diff)
    )

    mlflow.log_metric(
        "age_disparate_impact_ratio",
        float(age_di)
    )

    mlflow.log_metric(
        "gender_dp_difference",
        float(gender_dp_diff)
    )

    mlflow.log_metric(
        "gender_disparate_impact_ratio",
        float(gender_di)
    )

    # Spending-tier operational metric
    mlflow.log_metric(
        "spending_tier_dp_difference",
        float(spending_dp_diff)
    )

    # $50-$250 operational blind spot
    mlflow.log_metric(
        "blind_spot_recall_50_250",
        float(blind_spot_recall)
    )

    print(
        f"MLflow run completed: {run.info.run_id}"
    )

MLflow run completed: 24f9007b646f4dbcbab6b1a678928c9f
🏃 View run S402_Fairness_ at: https://mlflow.sagemaker.ap-southeast-1.app.aws/#/experiments/4/runs/24f9007b646f4dbcbab6b1a678928c9f
🧪 View experiment at: https://mlflow.sagemaker.ap-southeast-1.app.aws/#/experiments/4


## 11. Final Audit Summary

This audit evaluated Model A's held-out test predictions — at the
threshold recorded in Model A's own baseline manifest, not a value fixed
in this notebook — against age, gender, and spending-tier groupings.

**Demographic findings:** Age shows a real, multi-metric disparity —
Disparate Impact ratio 0.589 (below the 0.80 screening threshold),
corroborated by a substantial recall gap (53.33% to 81.06% across
brackets) and precision gap (72.07% to 96.92%). This mirrors a pattern
EDA already found in the raw fraud-rate data, now shown to carry through
into the model's actual detection behaviour. Gender shows a smaller,
currently-unflagged gap (Disparate Impact ratio 0.852) worth continued
monitoring rather than immediate action.

**Operational findings:** spending tier shows the intended, sharp
escalation in scrutiny by transaction amount — not a fairness concern.
Within that pattern, the `$50-$250` band is a specific, real blind spot:
zero recall against 61 actual fraud cases, distinct from the tier-level
pattern and worth investigating on its own.

**What this audit does not resolve:** whether the age disparity reflects
a defensible, data-driven risk difference or a pattern that should be
mitigated is a governance judgement, not a statistical one — this
notebook surfaces the signal; it does not decide what should be done
about it. Region/state fairness remains entirely unassessed given the
current feature set.

**Recommended next step:** treat the age finding as the priority item for
governance review, given it is corroborated across three independent
metrics rather than resting on a single number.